In [ ]:
import pandas as pd
import pm4py

from ocpm_partial_order.config import (
    MAIN_DATASET_DB,
    TABLES_DIR,
)
from ocpm_partial_order.io.ocel_loader import (
    load_ocel2_sqlite,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)

ocel = load_ocel2_sqlite(MAIN_DATASET_DB)

print("Dataset:", MAIN_DATASET_DB.name)
print(ocel)

In [ ]:
print("Colonne eventi:")
print(ocel.events.columns.tolist())

print("\nColonne oggetti:")
print(ocel.objects.columns.tolist())

print("\nColonne relazioni evento-oggetto:")
print(ocel.relations.columns.tolist())

In [ ]:
print("EVENTI")
display(ocel.events.head(10))

print("OGGETTI")
display(ocel.objects.head(10))

print("RELAZIONI EVENTO-OGGETTO")
display(ocel.relations.head(10))

In [ ]:
required_event_columns = {
    "ocel:eid",
    "ocel:activity",
    "ocel:timestamp",
}

required_object_columns = {
    "ocel:oid",
    "ocel:type",
}

required_relation_columns = {
    "ocel:eid",
    "ocel:oid",
    "ocel:type",
}

missing_event_columns = (
    required_event_columns - set(ocel.events.columns)
)
missing_object_columns = (
    required_object_columns - set(ocel.objects.columns)
)
missing_relation_columns = (
    required_relation_columns - set(ocel.relations.columns)
)

print("Colonne mancanti negli eventi:", missing_event_columns)
print("Colonne mancanti negli oggetti:", missing_object_columns)
print("Colonne mancanti nelle relazioni:", missing_relation_columns)

In [ ]:
general_summary = pd.DataFrame(
    [
        {
            "dataset": "Order Management",
            "events": len(ocel.events),
            "objects": len(ocel.objects),
            "event_object_relations": len(ocel.relations),
            "event_types": ocel.events[
                "ocel:activity"
            ].nunique(),
            "object_types": ocel.objects[
                "ocel:type"
            ].nunique(),
        }
    ]
)

display(general_summary)

In [ ]:
general_summary.to_csv(
    TABLES_DIR / "order_management_general_summary.csv",
    index=False,
)

In [ ]:
event_type_counts = (
    ocel.events["ocel:activity"]
    .value_counts()
    .rename_axis("activity")
    .reset_index(name="event_count")
)

display(event_type_counts)

In [ ]:
event_type_counts.to_csv(
    TABLES_DIR / "order_management_event_types.csv",
    index=False,
)

In [ ]:
object_type_counts = (
    ocel.objects["ocel:type"]
    .value_counts()
    .rename_axis("object_type")
    .reset_index(name="count")
)

display(object_type_counts)

In [ ]:
object_type_counts.to_csv(
    TABLES_DIR / "order_management_object_types.csv",
    index=False,
)

In [ ]:
objects_per_event = (
    ocel.relations
    .groupby("ocel:eid")["ocel:oid"]
    .nunique()
    .rename("object_count")
    .sort_values(ascending=False)
)

display(objects_per_event.head(20))

In [ ]:
display(objects_per_event.describe())

In [ ]:
multi_object_events = objects_per_event[
    objects_per_event > 1
]

print(
    "Eventi associati a più di un oggetto:",
    len(multi_object_events),
)

In [ ]:
object_types_per_event = (
    ocel.relations
    .groupby("ocel:eid")["ocel:type"]
    .nunique()
    .rename("object_type_count")
    .sort_values(ascending=False)
)

multi_type_events = object_types_per_event[
    object_types_per_event > 1
]

print(
    "Eventi associati a più object type:",
    len(multi_type_events),
)

display(multi_type_events.head(20))

In [ ]:
example_event_id = multi_type_events.index[0]

event_information = ocel.events[
    ocel.events["ocel:eid"] == example_event_id
]

event_relations = ocel.relations[
    ocel.relations["ocel:eid"] == example_event_id
]

display(event_information)
display(event_relations)

In [ ]:
activities_by_object_type = (
    pm4py.ocel_object_type_activities(ocel)
)

for object_type, activities in activities_by_object_type.items():
    print(f"\n{object_type}")
    for activity in sorted(activities):
        print(" -", activity)

## Conclusioni

Il dataset Order Management è un OCEL 2.0 contenente eventi
associati a più oggetti e a più tipi di oggetto.

I tipi principali sono orders, items, packages, customers,
employees e products.

La presenza di eventi condivisi tra tipi differenti conferma
che il processo non può essere rappresentato fedelmente tramite
un solo case identifier tradizionale.